<a href="https://colab.research.google.com/github/vasanthiginkala5/Advanced-Gen-AI-internship-Task-submission/blob/main/AI_Resume_Screening_System_with_Tracing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STRONG RESUME
strong_resume = """
Name: John
Skills: Python, Machine Learning, NLP, Deep Learning, SQL
Experience: 3 years Data Scientist
Tools: TensorFlow, PyTorch
"""

# AVERAGE RESUME
average_resume = """
Name: Jane
Skills: Python, Data Analysis, SQL
Experience: 1 year Data Analyst
Tools: Pandas, Excel
"""

# WEAK RESUME
weak_resume = """
Name: Alex
Skills: MS Word, Excel
Experience: Fresher
Tools: Basic Computer Knowledge
"""

# JOB DESCRIPTION
job_description = """
Role: Data Scientist

Requirements:
Python, Machine Learning, NLP, SQL,
TensorFlow or PyTorch experience
"""

In [ ]:
!pip install langchain langchain-community langchain-core -q

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
from langchain_core.prompts import PromptTemplate

In [ ]:
pipe = pipeline("text-generation", model="gpt2")

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
extract_prompt = PromptTemplate.from_template(
"""
Extract skills, experience, and tools from this resume:

{resume}

Do NOT assume anything.
"""
)

def extract(resume):
    return llm.invoke(extract_prompt.format(resume=resume))

In [ ]:
match_prompt = PromptTemplate.from_template(
"""
Compare this resume with job description.

Resume:
{resume}

Job:
{job}

Give:
- Matching skills
- Missing skills
"""
)

def match(resume, job):
    return llm.invoke(match_prompt.format(resume=resume, job=job))

In [ ]:
score_prompt = PromptTemplate.from_template(
"""
Give a score from 0–100 based on how well resume matches job.

Resume:
{resume}

Job:
{job}
"""
)

def score(resume, job):
    return llm.invoke(score_prompt.format(resume=resume, job=job))

In [ ]:
explain_prompt = PromptTemplate.from_template(
"""
Explain why this candidate got this score.

Resume:
{resume}

Job:
{job}
"""
)

def explain(resume, job):
    return llm.invoke(explain_prompt.format(resume=resume, job=job))

In [ ]:
print("🔹 STRONG CANDIDATE")
print(score(strong_resume, job_description))
print(explain(strong_resume, job_description))

print("\n🔹 AVERAGE CANDIDATE")
print(score(average_resume, job_description))
print(explain(average_resume, job_description))

print("\n🔹 WEAK CANDIDATE")
print(score(weak_resume, job_description))
print(explain(weak_resume, job_description))

In [ ]:
from langchain_core.prompts import PromptTemplate

# Extract Prompt
extract_prompt = PromptTemplate.from_template("""
Extract skills, experience, and tools from this resume:
{resume}
Do NOT assume anything.
""")

# Match Prompt
match_prompt = PromptTemplate.from_template("""
Compare resume with job description.

Resume:
{resume}

Job:
{job}

Give:
- Matching skills
- Missing skills
""")

# Score Prompt
score_prompt = PromptTemplate.from_template("""
Give a score (0-100) based on matching.

Resume:
{resume}

Job:
{job}
""")

# Explain Prompt
explain_prompt = PromptTemplate.from_template("""
Explain why this score was given.

Resume:
{resume}

Job:
{job}
""")

In [ ]:
# LCEL pipelines
extract_chain = extract_prompt | llm
match_chain = match_prompt | llm
score_chain = score_prompt | llm
explain_chain = explain_prompt | llm

In [ ]:
def run_pipeline(resume, job):
    print("\n--- Extract ---")
    extract_result = extract_chain.invoke({"resume": resume})
    print(extract_result)

    print("\n--- Match ---")
    match_result = match_chain.invoke({"resume": resume, "job": job})
    print(match_result)

    print("\n--- Score ---")
    score_result = score_chain.invoke({"resume": resume, "job": job})
    print(score_result)

    print("\n--- Explanation ---")
    explain_result = explain_chain.invoke({"resume": resume, "job": job})
    print(explain_result)

In [ ]:
print("🔹 STRONG")
run_pipeline(strong_resume, job_description)

print("\n🔹 AVERAGE")
run_pipeline(average_resume, job_description)

print("\n🔹 WEAK")
run_pipeline(weak_resume, job_description)

In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your_api_key"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [ ]:
bad_resume = "This person is expert in cooking and dancing"

In [ ]:
print("\n🔹 DEBUG CASE")
run_pipeline(bad_resume, job_description)

In [ ]:
extract_prompt = PromptTemplate.from_template("""
You are an AI resume analyzer.

Extract ONLY the information explicitly present in the resume.

Return in this format:
Skills: <list>
Experience: <text>
Tools: <list>

Rules:
- Do NOT assume or add any skill
- Do NOT hallucinate
- If information is missing, write "Not mentioned"

Resume:
{resume}
""")

In [ ]:
match_prompt = PromptTemplate.from_template("""
Compare the resume with the job description.

Return:
Matching Skills: <list>
Missing Skills: <list>

Rules:
- Only compare based on given text
- Do NOT assume skills
- Be strict in matching

Resume:
{resume}

Job Description:
{job}
""")

In [ ]:
score_prompt = PromptTemplate.from_template("""
Evaluate the resume against the job description.

Give a score between 0 and 100.

Rules:
- High match → high score
- Missing important skills → reduce score
- No guessing

Return only:
Score: <number>

Resume:
{resume}

Job:
{job}
""")

In [ ]:
explain_prompt = PromptTemplate.from_template("""
Explain why this score was assigned.

Include:
- Strengths
- Weaknesses
- Missing skills

Rules:
- Do NOT assume anything
- Base explanation only on given data

Resume:
{resume}

Job:
{job}
""")

In [ ]:
from langchain_core.prompts import PromptTemplate

few_shot_prompt = PromptTemplate.from_template("""
You are an AI resume evaluator.

Example:
Resume: Python, ML, NLP
Job: Python, ML
Score: 85

Example:
Resume: Excel, Word
Job: Python, ML
Score: 30

Now evaluate:

Resume:
{resume}

Job:
{job}

Return only:
Score: <number>
""")

few_shot_chain = few_shot_prompt | llm

print(few_shot_chain.invoke({
    "resume": strong_resume,
    "job": job_description
}))

In [ ]:
json_prompt = PromptTemplate.from_template("""
Extract details from resume.

Return ONLY in JSON format:

{{
  "skills": [],
  "experience": "",
  "tools": []
}}

Rules:
- Do NOT add extra text
- Do NOT assume

Resume:
{resume}
""")

json_chain = json_prompt | llm

print(json_chain.invoke({"resume": strong_resume}))

In [ ]:
extract_chain.invoke(
    {"resume": strong_resume},
    config={"tags": ["strong_candidate", "extract"]}
)

score_chain.invoke(
    {"resume": strong_resume, "job": job_description},
    config={"tags": ["strong_candidate", "score"]}
)

In [ ]:
def pipeline(resume, job):
    extract = extract_chain.invoke({"resume": resume})
    match = match_chain.invoke({"resume": resume, "job": job})
    score = score_chain.invoke({"resume": resume, "job": job})
    explain = explain_chain.invoke({"resume": resume, "job": job})

    print("Extract:", extract)
    print("Match:", match)
    print("Score:", score)
    print("Explanation:", explain)